<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/phonchi/CryoParticleSegment/blob/main/notebook/03_select_hyperparam_for_extraction_clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
</table>

### CryoParticleSegment

In [1]:
do = False # @param{type:"boolean"}
if do :
    %pip install torchinfo -qq
    %pip install -U git+https://github.com/qubvel/segmentation_models.pytorch -qq
    %pip install starfile -qq
    %pip install https://github.com/soft-matter/trackpy/archive/master.zip -qq

> #### ⚠ Notice
>
> You need to restart the kernel after the following step.

In [2]:
if do :
    %pip install pycuda==2024.1
    %pip install "numpy<2.0"
    %pip install mrcfile -qq

## ⭐ Setup
You must run all codes under this category.

### ✅ Directory Settings

In [3]:
# @title  { display-mode: "form" }

INPUT_IMAGE_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output_rst/dataset/10017/processed_micrographs_np_split" # @param {type:"string"}
IMAGE_DIR = INPUT_IMAGE_DIR
# @markdown ---

use_denoised_as_pariwise = False # @param {type : "boolean"}
dnzd_pw = use_denoised_as_pariwise
DENOISED_IMAGE_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output_rst/dataset/10017/processed_micrographs_np_split" # @param {type:"string"}
# @markdown ---
LABEL_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output/dataset/10017/micrographs_ground_np" # @param {type:"string"}
DATASET_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output/dataset" # @param {type:"string"}
RESULT_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output_rst/results_watershed_clustering/10017/unet_eb5_dice_CRF" # @param {type:"string"}

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
# @title  { display-mode: "form" }
# @markdown Detect whether using folder in Google Drive as **`RESULT DIR`**📁.
import os
if "content" in IMAGE_DIR.split("/")[:3] or "content" in LABEL_DIR.split("/")[:3]:
  try:
    from google.colab import drive
    drive.mount('/content/drive')
    !rm -r /content/sample_data
    if not os.path.exists("/content/image_dir"):
        if "content" in IMAGE_DIR.split("/")[:3]:
            !cp -r {IMAGE_DIR} /content/image_dir
            IMAGE_DIR = "/content/image_dir"
        if "content" in LABEL_DIR.split("/")[:3]:
            !cp -r {LABEL_DIR} /content/label_dir
            LABEL_DIR = "/content/label_dir"
  except:
    pass

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
rm: cannot remove '/content/sample_data': No such file or directory


In [6]:
IMAGE_DIR = "/content/image_dir"

In [7]:
!git clone https://github.com/phonchi/CryoParticleSegment.git

!wget -O /content/CryoParticleSegment/Modeling/convcrf.py https://raw.githubusercontent.com/SRT-0/CPS_modeling_adjusted_for_denoise_CRF/main/adjusted_modeling/convcrf.py
!wget -O /content/CryoParticleSegment/Modeling/dataset.py https://raw.githubusercontent.com/SRT-0/CPS_modeling_adjusted_for_denoise_CRF/main/adjusted_modeling/dataset.py
!wget -O /content/CryoParticleSegment/Modeling/model.py https://raw.githubusercontent.com/SRT-0/CPS_modeling_adjusted_for_denoise_CRF/main/adjusted_modeling/model.py
!wget -O /content/CryoParticleSegment/Modeling/trainer.py https://raw.githubusercontent.com/SRT-0/CPS_modeling_adjusted_for_denoise_CRF/main/adjusted_modeling/trainer.py

Cloning into 'CryoParticleSegment'...
remote: Enumerating objects: 270, done.
remote: Counting objects: 100% (270/270), done.
remote: Compressing objects: 100% (253/253), done.
remote: Total 270 (delta 141), reused 42 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (270/270), 32.01 MiB | 12.12 MiB/s, done.
Resolving deltas: 100% (141/141), done.
--2026-01-13 02:30:31--  https://raw.githubusercontent.com/SRT-0/CPS_modeling_adjusted_for_denoise_CRF/main/adjusted_modeling/convcrf.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 17753 (17K) [text/plain]
Saving to: ‘/content/CryoParticleSegment/Modeling/convcrf.py’

/content/CryoPartic 100%[===================>]  17.34K  --.-KB/s    in 0.001s  

2026-01-13 02:30:32 (22.0 MB/s) - ‘/content/CryoParticl

In [8]:
import sys
import os

# Adjust the path relative to your current working directory
module_path = os.path.abspath('CryoParticleSegment/Modeling')

# Add to sys.path if it's not already included
if module_path not in sys.path:
    sys.path.append(module_path)

### ✅ Packages Handling

In [9]:
# @title  { display-mode: "form" }
# @markdown Useful packages.

import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import torch
from torch.utils.data import DataLoader
from torchvision import transforms

In [10]:
# @title  { display-mode: "form" }
# @markdown User-defined packages.

from dataset import MicrographDataset, MicrographDatasetEvery, collate_fn

## ⭐ Main

### ✅ Setting

In [11]:
# @markdown Parameters.

user = True # @param {type:"boolean"}

In [12]:
# @markdown Parameters.

BATCH = 8
CROP_SIZE = (512, 512)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [13]:
# @markdown Set seed.

random_state = 42
torch.manual_seed(random_state)
torch.cuda.manual_seed_all(random_state)

### ✅ Dataset

You can provide a [`transforms.CenterCrop(3840)`](https://docs.pytorch.org/vision/master/generated/torchvision.transforms.CenterCrop.html) object to crop out boundary artifacts.


In [14]:
crop = transforms.CenterCrop(3840)

In [15]:
train_dir = os.path.join(IMAGE_DIR, 'train')
train_filenames = np.loadtxt(f"{IMAGE_DIR}/train_filenames.txt", dtype=str)
if dnzd_pw == False:
    train_dataset = MicrographDataset(image_dir=train_dir, label_dir=LABEL_DIR, filenames=train_filenames, crop_size=CROP_SIZE, num_patches = 4, crop=crop)
else:
    dnzd_train_dir = os.path.join(DENOISED_IMAGE_DIR, 'train')
    train_dataset = MicrographDataset(image_dir=train_dir, label_dir=LABEL_DIR, denoised_dir = dnzd_train_dir, filenames=train_filenames, crop_size=CROP_SIZE, num_patches = 4, crop=crop)

In [16]:
val_dir = os.path.join(IMAGE_DIR, 'val')
val_filenames = np.loadtxt(f"{IMAGE_DIR}/val_filenames.txt", dtype=str)
if dnzd_pw == False:
    val_dataset = MicrographDatasetEvery(image_dir=val_dir, label_dir=LABEL_DIR, filenames=val_filenames, crop_size=CROP_SIZE, crop=crop)
else:
    dnzd_val_dir = os.path.join(DENOISED_IMAGE_DIR, 'val')
    val_dataset = MicrographDatasetEvery(image_dir=val_dir, label_dir=LABEL_DIR, denoised_dir = dnzd_val_dir, filenames=val_filenames, crop_size=CROP_SIZE, crop=crop)
val_loader = DataLoader(val_dataset, batch_size=None, shuffle=False, pin_memory=True, collate_fn=collate_fn)

In [17]:
if not user:
    test_dir = os.path.join(IMAGE_DIR, 'test')
    test_filenames = np.loadtxt(f"{IMAGE_DIR}/test_filenames.txt", dtype=str)
    if dnzd_pw == False:
        test_dataset = MicrographDatasetEvery(image_dir=test_dir, label_dir=LABEL_DIR, filenames=test_filenames, crop_size=CROP_SIZE)
    else:
        dnzd_test_dir = os.path.join(DENOISED_IMAGE_DIR, 'test')
        test_dataset = MicrographDatasetEvery(image_dir=test_dir, label_dir=LABEL_DIR, denoised_dir = dnzd_test_dir, filenames=test_filenames, crop_size=CROP_SIZE)

    test_loader = DataLoader(test_dataset, batch_size=None, shuffle=False, pin_memory=True, collate_fn=collate_fn)

In [18]:
for i1, i2, i3, i4, i5 in val_loader: #test loader and reconstruct
    print(i3.dtype, i5.dtype)
    print(i3.shape, i5.shape)
    shape = i5.shape
    break

torch.int64 torch.int64
torch.Size([81, 1, 512, 512]) torch.Size([1, 3840, 3840])


## ⭐ Evaluate

In [19]:
import gc
gc.collect()
torch.cuda.empty_cache()

from torchvision.utils import save_image
import starfile
import pandas as pd
import matplotlib
from PIL import Image
import cv2

def get_basename_with_uid_removed(path):
  return os.path.basename(path).split(sep='_', maxsplit=1)[-1]


def simple_micrograph_preprocessing(micrograph):
  micrograph_copy = micrograph.copy()
  micrograph_copy = (micrograph_copy-micrograph.mean()+2.5*micrograph.std())/5/micrograph.std()
  micrograph_copy[micrograph_copy<0]=0
  micrograph_copy[micrograph_copy>1]=1
  return micrograph_copy


In [20]:
# @title  { vertical-output: true, display-mode: "form" }
EMPIAR_ID = 10017 # @param {type:"integer"}
RADIUS = 64 # @param {type:"integer"}
# For 10017
BORDER = 128 # @param {type:"integer"}
SIZE = 4096 # @param {type:"integer"}

In [21]:
!cp {DATASET_DIR}/{EMPIAR_ID}/filtered_val.star .

In [22]:
y_size = SIZE
labeled_particles = starfile.read(f"filtered_val.star")['particles']
labeled_particles = labeled_particles[['rlnMicrographName', 'rlnCoordinateX', 'rlnCoordinateY']]
labeled_particles.columns = pd.Index(['image_name', 'x_coord', 'y_coord'])
labeled_particles['image_name'] = labeled_particles['image_name'].apply(get_basename_with_uid_removed)
labeled_particles['image_name'] = labeled_particles['image_name'].apply(lambda s: s.split(".")[0])
labeled_particles['y_coord'] = y_size - labeled_particles['y_coord']
labeled_particles

,image_name,x_coord,y_coord
0,Falcon_2012_06_13-03_22_02_0,2169,1426
1,Falcon_2012_06_13-03_22_02_0,2791,1957
2,Falcon_2012_06_13-03_22_02_0,2372,475
3,Falcon_2012_06_13-03_22_02_0,2635,1047
4,Falcon_2012_06_13-03_22_02_0,3560,3965
...,...,...,...
3540,Falcon_2012_06_12-15_14_01_0,1810,1593
3541,Falcon_2012_06_12-15_14_01_0,1178,1780
3542,Falcon_2012_06_12-15_14_01_0,364,1047
3543,Falcon_2012_06_12-15_14_01_0,961,556


In [23]:
def preprocess_and_crop(micrograph, crop_size=3840):
    processed_micrograph = simple_micrograph_preprocessing(micrograph)
    if crop_size:
        mic_width, mic_height = processed_micrograph.shape[1], processed_micrograph.shape[0]
        start_x, start_y = (mic_width - crop_size) // 2, (mic_height - crop_size) // 2
        end_x, end_y = start_x + crop_size, start_y + crop_size
        return processed_micrograph[start_y:end_y, start_x:end_x]
    else:
        return processed_micrograph

def plot_micrograph_and_labels(ax, micrograph, labels, coords):
    ax.imshow(micrograph, cmap='gray')
    ax.imshow(labels, cmap='gray', alpha=0.5)
    for x, y in coords:
        corrected_x, corrected_y = x, y
        circle = matplotlib.patches.Circle((corrected_x, corrected_y), radius=RADIUS, fill=False, color='r')
        ax.add_patch(circle)

You can specify a `crop_size` in `preprocess_and_crop()` to remove boundary artifacts during preprocessing.

In [24]:
label_images = np.empty((0, shape[1], shape[2]), dtype=np.uint8)
gts = []

for idx, (test_image, _, _, grid, _) in enumerate(val_dataset):
    # if idx == 6:
    #     break
    name = val_filenames[idx][:-4]
    micrograph = np.load(f"{IMAGE_DIR}/val/{name}.npy")
    label_path = f"{LABEL_DIR}/{name}.png"
    image = Image.open(label_path)
    label_image = np.array(image)

    cropped_micrograph = preprocess_and_crop(micrograph)
    cropped_label_image = preprocess_and_crop(label_image)
    print(cropped_label_image.shape)
    label_images = np.concatenate((label_images, [cropped_label_image]), axis=0)

    locations = labeled_particles[labeled_particles['image_name'] == name]
    _, ax = plt.subplots(figsize=(12, 12))
    coords = locations[['x_coord', 'y_coord']].values - BORDER
    plot_micrograph_and_labels(ax, cropped_micrograph, cropped_label_image, coords)
    plt.show()
    print(len(coords))
    gts.append(coords)
    ##

#filename = f"{os.path.splitext(checkpoint_path)[0]}.png"
#pred_path = os.path.join(RESULT_DIR, "Each_ckpt", filename)
#save_image(pred_image, pred_path)

Output hidden; open in https://colab.research.google.com to view.

In [25]:
label_images.shape

(6, 3840, 3840)

### Score

> #### 🗒 Info
> Here, we compute the score based on the validation set. You may choose to rank the algorithms using either the F-score or mAP. Additionally, the parameter $\beta$ in the F-score can be adjusted: values of $\beta > 1$ place greater emphasis on recall over precision, while $\beta < 1$ give more weight to precision.


In [26]:
from metrics import centers_to_boxes, calculate_iou_torchvision, evaluate_detection_raw_multiple, f_beta_score, calculate_mAP_multiple_images

In [27]:
# Assign a default confidence score of 1.0 to all predicted boxes
default_score = 1.0
beta = 1 # @param {type:"number"}
F_score = False # @param {type:"boolean"}

---
## DT with circularity, Sphericity, Slenderness at scalar for the mean of front 50% 0.65

In [28]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

def plot_clustered_particles(micrograph, prob_map, region_data, cluster_labels, particle_radius=64, title="Particles by Cluster"):

    fig, ax = plt.subplots(figsize=(14, 14))

    # 1. Base Micrograph layer
    ax.imshow(micrograph, cmap='gray')

    # 2. Probability Mask Overlay
    ax.imshow(prob_map, cmap='gray', alpha=0.4)

    unique_clusters = np.unique(cluster_labels)
    colors = plt.cm.get_cmap('tab10', len(unique_clusters))
    legend_patches = []

    for idx, cluster_id in enumerate(unique_clusters):
        indices = np.where(cluster_labels == cluster_id)[0]
        color = colors(idx)

        for i in indices:
            region = region_data[i]
            cy, cx = region.centroid

            # 3. Particle Boundary Ring
            boundary = mpatches.Circle((cx, cy), radius=particle_radius, fill=False, color=color, alpha=0.6, linewidth=1.5)
            ax.add_patch(boundary)

            # 4. Cluster ID Text Label
            ax.text(cx, cy, str(cluster_id), color='white', fontsize=9,
                    ha='center', va='center', fontweight='bold',
                    bbox=dict(facecolor=color, alpha=0.3, edgecolor='none', boxstyle='round,pad=0.2'))

        legend_patches.append(mpatches.Patch(color=color, label=f'Cluster {cluster_id}'))

    ax.set_title(title, fontsize=20)
    ax.legend(handles=legend_patches, loc='upper right', title="Cluster IDs")
    ax.axis('off')
    plt.tight_layout()
    plt.show()

In [29]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math
import umap
import seaborn as sns
from scipy.spatial.distance import mahalanobis
from scipy.ndimage import rotate
from sklearn.mixture import BayesianGaussianMixture
from sklearn.preprocessing import StandardScaler
from skimage.segmentation import watershed
from skimage.feature import peak_local_max
from skimage.measure import regionprops
from scipy import ndimage as ndi
from IPython.display import display

def get_shape_metrics(region):
    """Calculates custom shape descriptors for particle analysis."""
    def _circularity_(region):
        area = region.area
        perimeter = region.perimeter
        return (4 * np.pi * area) / (perimeter ** 2) if perimeter > 0 else 0

    def _eccentricity_(region):
        return region.eccentricity

    def _solidity_(region):
        return region.area / region.convex_area if region.convex_area > 0 else 0

    metrics_to_run = [_circularity_, _eccentricity_, _solidity_]
    return tuple(func(region) for func in metrics_to_run), region.area

def Watershed_DT_cluster(micrograph_input, prob_map_input,
                        particle_radius=64,
                        peak_threshold_ratio=0.6, min_dist_ratio=0.4,
                        show_plots=True, show_table=False, graph_name=None):

    # 1. Normalization & Masking
    micrograph = micrograph_input.astype(np.float32)
    prob_map = prob_map_input.astype(np.float32)
    height, width = prob_map.shape[:2]
    if prob_map.max() > 1.0: prob_map /= 255.0

    binary_mask = (prob_map > 0.5).astype(np.uint8)
    if np.sum(binary_mask) == 0: binary_mask = (prob_map > 0.05).astype(np.uint8)
    if np.sum(binary_mask) == 0: return []

    # 2. Watershed Segmentation
    #
    distance = ndi.distance_transform_edt(binary_mask)
    local_max_coords = peak_local_max(distance, min_distance=int(particle_radius * min_dist_ratio),
                                     labels=binary_mask, threshold_abs=particle_radius * peak_threshold_ratio)
    markers = np.zeros(distance.shape, dtype=int)
    for i, (r, c) in enumerate(local_max_coords): markers[r, c] = i + 1
    labels = watershed(-distance, markers, mask=binary_mask)
    regions = regionprops(labels)
    if not regions: return []

    # 3. Feature Extraction & Global Mahalanobis
    features, region_data = [], []
    for region in regions:
        shape_metrics, area = get_shape_metrics(region)
        features.append([area] + list(shape_metrics)) # Features: [Area, Circ, Ecc, Sol]
        region_data.append(region)
    X = np.array(features)

    center = np.median(X, axis=0)
    inv_cov = np.linalg.pinv(np.cov(X, rowvar=False))
    all_mahal_distances = np.array([mahalanobis(p, center, inv_cov) for p in X])

    # 4. Clustering (UMAP + BGMM)
    #
    X_scaled = StandardScaler().fit_transform(X)
    X_embedding = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2, random_state=42).fit_transform(X_scaled)
    bgmm = BayesianGaussianMixture(n_components=7, weight_concentration_prior=1e-2, random_state=42)
    cluster_labels = bgmm.fit_predict(X_embedding)

    # 5. Group-Level Outlier Identification
    unique_clusters = np.unique(cluster_labels)
    cluster_mahal_means = {c: np.mean(all_mahal_distances[cluster_labels == c]) for c in unique_clusters}
    outlier_cluster_id = max(cluster_mahal_means, key=cluster_mahal_means.get)

    # 6. Visualization and Statistics
    if show_plots:
        plot_clustered_particles(micrograph, prob_map, region_data, cluster_labels,
                                 particle_radius=particle_radius,
                                 title=f"Cluster Analysis & Mask: {graph_name}")
        plt.style.use('default')
        n_clusters = len(unique_clusters)

        # PART A: Row-based Cluster Comparison
        fig, axes = plt.subplots(n_clusters, 3, figsize=(18, 4 * n_clusters))
        if n_clusters == 1: axes = np.atleast_2d(axes)

        all_stats = []

        for idx, c_id in enumerate(unique_clusters):
            indices = np.where(cluster_labels == c_id)[0]
            current_n = len(indices)
            aligned_patches = []

            for i in indices:
                region = region_data[i]
                cy, cx = region.centroid
                angle_deg = np.degrees(region.orientation)

                tr, orr = int(particle_radius * 1.1), int(particle_radius * 1.6)
                y1, y2, x1, x2 = int(cy)-orr, int(cy)+orr, int(cx)-orr, int(cx)+orr

                if y1 > 0 and y2 < height and x1 > 0 and x2 < width:
                    # Aligned crop using Overscan Logic
                    patch = rotate(micrograph[y1:y2, x1:x2], -angle_deg, reshape=False, order=1)
                    aligned_patches.append(patch[orr-tr:orr+tr, orr-tr:orr+tr])

            if aligned_patches:
                mean_img = np.mean(aligned_patches, axis=0)
                dim = mean_img.shape[0]
                center_pt, step = dim // 2, (dim // 2) // 10

                # Concentric Binning logic
                bin_data = []
                shell_bounds = [] # Store coordinates for overlay
                for s in range(10):
                    r_out = (dim // 2) - (s * step)
                    r_in = (dim // 2) - ((s + 1) * step)
                    m_mask = np.zeros_like(mean_img, dtype=bool)
                    m_mask[center_pt-r_out:center_pt+r_out, center_pt-r_out:center_pt+r_out] = True
                    m_mask[center_pt-r_in:center_pt+r_in, center_pt-r_in:center_pt+r_in] = False
                    bin_data.append(mean_img[m_mask])
                    shell_bounds.append(r_out)

                bg_mean, bg_var = np.mean(bin_data[0]), np.var(bin_data[0])
                snr_profile = []

                for s in range(10):
                    curr_var = np.var(bin_data[s] - bg_mean)
                    snr_val = math.log10(curr_var / bg_var) if bg_var > 1e-12 else 0
                    snr_profile.append(snr_val)

                    all_stats.append({
                        'Cluster': c_id, 'N': current_n, 'Bin': s,
                        'Is_Outlier': "Yes" if c_id == outlier_cluster_id else "No",
                        'Mean': np.mean(bin_data[s]), 'Variance': np.var(bin_data[s]),
                        'SNR_vs_Bin0': snr_val
                    })

                # Plotting Columns
                axes[idx, 0].imshow(mean_img, cmap='gray')
                axes[idx, 0].set_title(f"C{c_id} Mean (N={current_n}) {'[OUTLIER]' if c_id == outlier_cluster_id else ''}")

                for rb in shell_bounds:
                    rect = plt.Rectangle((center_pt - rb, center_pt - rb), 2*rb, 2*rb,
                                         fill=False, color='red', alpha=0.3, linewidth=1, linestyle = "--")
                    axes[idx, 0].add_patch(rect)
                axes[idx, 0].axis('off')

                axes[idx, 1].bar(range(10), snr_profile, color='black', alpha=0.7)
                axes[idx, 1].set_title("SNR Profile (Outside -> Inside)", color='black')
                axes[idx, 1].set_xticks(range(10))

                axes[idx, 2].scatter(X_embedding[:,0], X_embedding[:,1], c='lightgray', alpha=0.2, s=5)
                axes[idx, 2].scatter(X_embedding[indices,0], X_embedding[indices,1], c='darkcyan', s=15)
                axes[idx, 2].set_title(f"UMAP Highlight | Mahal Mean: {cluster_mahal_means[c_id]:.2f}", color='black')
                axes[idx, 2].axis('off')

        plt.suptitle(f"Cluster Analysis (Graph: {graph_name})", fontsize=16, color='black')
        plt.tight_layout(rect=[0, 0.03, 1, 0.97])
        plt.show()

        # PART B: Metric Violin Plots
        #
        metrics_df = pd.DataFrame(X, columns=['Area', 'Circularity', 'Eccentricity', 'Solidity'])
        metrics_df['Mahalanobis'] = all_mahal_distances
        metrics_df['Cluster'] = cluster_labels

        v_cols = ['Area', 'Circularity', 'Eccentricity', 'Solidity', 'Mahalanobis']
        fig, v_axes = plt.subplots(1, 5, figsize=(25, 5))
        for i, var in enumerate(v_cols):
            sns.violinplot(x='Cluster', y=var, data=metrics_df, ax=v_axes[i],
                           palette='muted', hue='Cluster', legend=False)
            v_axes[i].set_title(f'{var} Dist')
            v_axes[i].grid(axis='y', linestyle='--', alpha=0.3)
        plt.show()

        # UMAP Scatter Plot
        plt.figure(figsize=(12, 4))
        scatter = plt.scatter(X_embedding[:, 0], X_embedding[:, 1], c=cluster_labels, cmap='viridis', s=15, alpha=0.6)
        plt.title(f"UMAP Space | Global Avg Mahal Dist: {np.mean(list(cluster_mahal_means.values())):.2f}", color='black')
        plt.colorbar(scatter, label='Cluster ID')
        plt.show()

        # PART C: Data Table
        if show_table:
            df = pd.DataFrame(all_stats)
            styled_table = df.style.set_table_attributes('style="font-size: 15px; width: 100%; color: black;"')\
                                .background_gradient(subset=['SNR_vs_Bin0'], cmap='plasma')\
                                .format({'Mean': '{:.4f}', 'Variance': '{:.4f}', 'SNR_vs_Bin0': '{:.4f}'})
            display(styled_table)

    # 7. Final Selection (Excluding outliers, selecting best median circularity/solidity)
    valid_particles = []
    return valid_particles

In [30]:
import itertools

watershed_list_all = []
watershed_config = []

# Search Space
threshold_levels_ratios = [0.4]
min_dist_ratios = [0.8]

print(f"Starting DT-Only Grid Search with BGMM Clustering and Mean Averaging...")

param_grid = list(itertools.product(
    threshold_levels_ratios, min_dist_ratios
))

# Iterate through hyperparameters
for params in param_grid:
    thresh_r, dist_r = params
    print(f"\n--- Testing Config: Thresh={thresh_r}, Dist={dist_r} ---")

    current_config_results = []

    # Iterate through validation dataset
    # changed) Now fetching both micrograph and label_image for averaging/clustering
    for idx in range(len(val_filenames)):
        name = val_filenames[idx][:-4]

        # 1. Load data
        print(f"\n[Processing Graph Index: {idx}] Micrograph: {name}")
        micrograph = np.load(f"{IMAGE_DIR}/val/{name}.npy")
        prob_map = label_images[idx] # Assuming label_images contains the predicted prob maps

        # 2. Match dimensions (preprocessing)
        cropped_micrograph = preprocess_and_crop(micrograph)

        # 3. Perform Clustering and Particle Picking
        # changed) show_plots set to False during grid search to prevent 100s of popups
        particles = Watershed_DT_cluster(
            micrograph_input=cropped_micrograph,
            prob_map_input=prob_map,
            particle_radius=RADIUS,
            peak_threshold_ratio=thresh_r,
            min_dist_ratio=dist_r,
            show_plots=True,
            graph_name = name
        )
        current_config_results.append(particles)

    watershed_list_all.append(current_config_results)
    watershed_config.append((thresh_r, dist_r))

print(f"\nDone! Generated {len(watershed_list_all)} hyperparameter candidates.")

Output hidden; open in https://colab.research.google.com to view.

---

## class

In [50]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import math
import umap
from sklearn.mixture import BayesianGaussianMixture
from sklearn.preprocessing import StandardScaler
from skimage.segmentation import watershed
from skimage.feature import peak_local_max
from skimage.measure import regionprops
from scipy import ndimage as ndi
from scipy.ndimage import rotate
from scipy.spatial.distance import mahalanobis
import random
from scipy.stats import levene
import warnings

def plot_clustered_particles(micrograph, prob_map, region_data, cluster_labels,
                             particle_radius=64, title="Particles by Cluster"):
    """
    Primary overlay plot: Micrograph + Prob Mask + Cluster Rings/IDs.
    """
    fig, ax = plt.subplots(figsize=(14, 14))
    ax.imshow(micrograph, cmap='gray')
    ax.imshow(prob_map, cmap='gray', alpha=0.4)

    unique_clusters = np.unique(cluster_labels)
    colors = plt.cm.get_cmap('tab10', len(unique_clusters))
    legend_patches = []

    for idx, cluster_id in enumerate(unique_clusters):
        indices = np.where(cluster_labels == cluster_id)[0]
        color = colors(idx)
        for i in indices:
            cy, cx = region_data[i].centroid
            boundary = mpatches.Circle((cx, cy), radius=particle_radius, fill=False,
                                        color=color, alpha=0.6, linewidth=1.5)
            ax.add_patch(boundary)
            ax.text(cx, cy, str(cluster_id), color='white', fontsize=9,
                    ha='center', va='center', fontweight='bold',
                    bbox=dict(facecolor=color, alpha=0.3, edgecolor='none', boxstyle='round,pad=0.2'))
        legend_patches.append(mpatches.Patch(color=color, label=f'Cluster {cluster_id}'))

    ax.set_title(title, fontsize=20)
    ax.legend(handles=legend_patches, loc='upper right', title="Cluster IDs")
    ax.axis('off')
    plt.tight_layout()
    plt.show()

class Watershed_Cluster():
    def __init__(self, particle_radius=64, n_clusters=7, seed=42, snr_std_thresh=0.15):
        self.particle_radius = particle_radius
        self.n_clusters = n_clusters
        self.seed = seed
        self.snr_std_thresh = snr_std_thresh

        self.metric_names = ['Area', 'Circularity', 'Eccentricity', 'Solidity', 'Hu_1', 'Hu_2', 'Hu_3']
        self.num_metrics = len(self.metric_names)
        random.seed(seed)
        warnings.filterwarnings("ignore")

    def get_shape_metrics(self, region):
        """Calculates custom descriptors and first 3 Hu Moments."""

        def _circularity_(region): return (4 * np.pi * region.area) / (region.perimeter ** 2) if region.perimeter > 0 else 0

        def _eccentricity_(region): return region.eccentricity

        def _solidity_(region): return region.area / region.convex_area if region.convex_area > 0 else 0

        base_metrics = [_circularity_(region), _eccentricity_(region), _solidity_(region)]

        hu_moments = list(region.moments_hu[:3])

        return tuple(base_metrics + hu_moments), region.area

    def _prepare_inputs(self, micrograph_input, prob_map_input):
        micrograph = micrograph_input.astype(np.float32)
        prob_map = prob_map_input.astype(np.float32)
        if prob_map.max() > 1.0: prob_map /= 255.0
        binary_mask = (prob_map > 0.5).astype(np.uint8)
        if np.sum(binary_mask) == 0:
            binary_mask = (prob_map > 0.05).astype(np.uint8)
        return micrograph, prob_map, binary_mask

    def _apply_watershed(self, binary_mask, peak_threshold_ratio, min_dist_ratio):
        distance = ndi.distance_transform_edt(binary_mask)
        local_max_coords = peak_local_max(
            distance, min_distance=int(self.particle_radius * min_dist_ratio),
            labels=binary_mask, threshold_abs=self.particle_radius * peak_threshold_ratio
        )
        markers = np.zeros(distance.shape, dtype=int)
        for i, (r, c) in enumerate(local_max_coords): markers[r, c] = i + 1
        labels = watershed(-distance, markers, mask=binary_mask)
        return regionprops(labels)

    def plot_diagnostics(self, micrograph, prob_map, region_data, unique_clusters,
                         cluster_outlier_status, cluster_mahal_means, X,
                         all_mahal_distances, cluster_labels, aligned_data,
                         X_embedding, graph_name):
        """
        - PART A: Row-based Cluster Comparison (Mean Img, SNR, UMAP Highlight)
        - PART B: Metric Violin Plots (Distribution of shape features)
        - PART C: Global UMAP Scatter Plot
        """
        plot_clustered_particles(micrograph, prob_map, region_data, cluster_labels,
                                 particle_radius=self.particle_radius,
                                 title=f"Cluster Analysis & Mask: {graph_name}")
        plt.style.use('default')
        n_clusters = len(unique_clusters)

        # --- PART A: Row-based Cluster Comparison ---
        fig, axes = plt.subplots(n_clusters, 3, figsize=(18, 4 * n_clusters))
        if n_clusters == 1: axes = np.atleast_2d(axes)

        for idx, c_id in enumerate(unique_clusters):
            indices = np.where(cluster_labels == c_id)[0]
            current_n = len(indices)

            # Identify outlier status for title
            status_val = cluster_outlier_status[c_id]
            is_out = status_val != "No"
            status_text = f"[{status_val}]" if is_out else ""

            # Column 0: Mean Image with concentric rings
            if c_id in aligned_data:
                mean_img, snr_profile, shell_bounds = aligned_data[c_id]
                dim = mean_img.shape[0]
                center_pt = dim // 2

                axes[idx, 0].imshow(mean_img, cmap='gray')
                axes[idx, 0].set_title(f"C{c_id} Mean (N={current_n}) {status_text}",
                                       color='red' if is_out else 'black')

                for rb in shell_bounds:
                    rect = plt.Rectangle((center_pt - rb, center_pt - rb), 2*rb, 2*rb,
                                         fill=False, color='red', alpha=0.3, linewidth=1, linestyle="--")
                    axes[idx, 0].add_patch(rect)
                axes[idx, 0].axis('off')

                # Column 1: SNR Profile
                axes[idx, 1].bar(range(10), snr_profile, color='black', alpha=0.7)
                axes[idx, 1].set_title("SNR Profile (Outside -> Inside)")
                axes[idx, 1].set_xticks(range(10))

                # Column 2: UMAP Highlight
                axes[idx, 2].scatter(X_embedding[:,0], X_embedding[:,1], c='lightgray', alpha=0.2, s=5)
                axes[idx, 2].scatter(X_embedding[indices,0], X_embedding[indices,1], c='darkcyan', s=15)
                axes[idx, 2].set_title(f"UMAP Highlight | Mahal Mean: {cluster_mahal_means[c_id]:.2f}")
                axes[idx, 2].axis('off')

        plt.suptitle(f"Cluster Analysis (Graph: {graph_name})", fontsize=16)
        plt.tight_layout(rect=[0, 0.03, 1, 0.97])
        plt.show()

        # --- PART B: Metric Violin Plots ---
        # Dataframe setup for Parts B and D
        metrics_df = pd.DataFrame(X, columns=self.metric_names)
        metrics_df['Mahalanobis'] = all_mahal_distances
        metrics_df['Cluster'] = cluster_labels

        # Determine grid size (3 columns wide)
        v_cols = self.metric_names + ['Mahalanobis']
        n_plots = len(v_cols)
        n_rows = math.ceil(n_plots / 3)
        # violin plot
        fig, v_axes = plt.subplots(n_rows, 3, figsize=(18, 5 * n_rows))
        v_axes = v_axes.flatten()

        for i, var in enumerate(v_cols):
            sns.violinplot(x='Cluster', y=var, data=metrics_df, ax=v_axes[i],
                           palette='muted', hue='Cluster', legend=False)
            v_axes[i].set_title(f'{var} Distribution')
            v_axes[i].grid(axis='y', linestyle='--', alpha=0.3)

        # Hide empty subplots
        for j in range(i + 1, len(v_axes)):
            v_axes[j].axis('off')

        plt.tight_layout(); plt.show()
        # --- PART C: Global UMAP Space ---
        plt.figure(figsize=(10, 6))
        avg_mahal = np.mean(list(cluster_mahal_means.values()))
        scatter = plt.scatter(X_embedding[:, 0], X_embedding[:, 1],
                             c=cluster_labels, cmap='viridis', s=15, alpha=0.6)
        plt.title(f"Global UMAP Space | Avg Mahal Dist: {avg_mahal:.2f}")
        plt.colorbar(scatter, label='Cluster ID')
        plt.show()

        # PART D: Histograms of each metric colored by cluster
        metrics_df = pd.DataFrame(X, columns=self.metric_names)
        metrics_df['Cluster'] = cluster_labels
        n_metrics = len(self.metric_names)
        rows = math.ceil(n_metrics / 3)
        fig, h_axes = plt.subplots(rows, 3, figsize=(18, 4 * rows))
        h_axes = h_axes.flatten()
        for i, var in enumerate(self.metric_names):
            sns.histplot(data=metrics_df, x=var, hue='Cluster', multiple="stack", ax=h_axes[i], palette='tab10', alpha=0.5)
            h_axes[i].set_title(f'{var} Stacked Histogram')
        for j in range(i + 1, len(h_axes)): h_axes[j].axis('off')
        plt.tight_layout(); plt.show()

    def _calculate_snr_profile(self, mean_img):
        dim = mean_img.shape[0]; cp, step = dim // 2, (dim // 2) // 10
        bin_data, shell_bounds = [], []
        for s in range(10):
            r_out, r_in = cp - (s * step), cp - ((s + 1) * step)
            m = np.zeros_like(mean_img, dtype=bool)
            m[cp-r_out:cp+r_out, cp-r_out:cp+r_out] = True
            m[cp-r_in:cp+r_in, cp-r_in:cp+r_in] = False
            bin_data.append(mean_img[m]); shell_bounds.append(r_out)
        bg_m, bg_v = np.mean(bin_data[0]), np.var(bin_data[0])
        snr_prof = [math.log10(np.var(b - bg_m) / bg_v) if bg_v > 1e-12 else 0 for b in bin_data]
        return snr_prof, shell_bounds

    def _evaluate_cluster_quality(self, c_id, indices, all_mahal_distances, X, cluster_labels, mahal_threshold):
        c_mahal_mean = np.mean(all_mahal_distances[indices])

        # Stat test for Hu_3 variance difference
        hu3_col = self.metric_names.index('Hu_3')
        cluster_hu3 = X[indices, hu3_col]
        others_hu3 = X[cluster_labels != c_id, hu3_col]

        # Levene's test is robust to non-normal distributions
        _, p_val = levene(cluster_hu3, others_hu3)
        significant_var_diff = p_val < 0.05

        if c_mahal_mean > mahal_threshold and significant_var_diff:
            return "OUTLIER", c_mahal_mean
        return "No", c_mahal_mean

    def process(self, micrograph_input, prob_map_input, peak_threshold_ratio=0.6,
                min_dist_ratio=0.4, N_from_clust=10, show_plots=True, graph_name=None):
        micrograph, prob_map, binary_mask = self._prepare_inputs(micrograph_input, prob_map_input)
        regions = self._apply_watershed(binary_mask, peak_threshold_ratio, min_dist_ratio)
        if not regions: return []

        features, region_data = [], []
        for r in regions:
            metrics, area = self.get_shape_metrics(r)
            features.append([area] + list(metrics)); region_data.append(r)
        X = np.array(features)

        # Robust Mahalanobis (MAD based)
        center = np.median(X, axis=0)
        precision = np.diag(1.0 / ((np.median(np.abs(X - center), axis=0) / 0.6745)**2 + 1e-6))
        all_mahal = np.array([mahalanobis(p, center, precision) for p in X])
        # Outlier boundary set strictly by Mahal MAD
        mahal_threshold = np.median(all_mahal) + (3.0 * np.median(np.abs(all_mahal - np.median(all_mahal))) / 0.6745)

        X_emb = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=self.seed).fit_transform(StandardScaler().fit_transform(X))
        cluster_labels = BayesianGaussianMixture(n_components=self.n_clusters, random_state=self.seed).fit_predict(X_emb)

        unique_clusters = np.unique(cluster_labels)
        cluster_outlier_status, cluster_mahal_means, aligned_data = {}, {}, {}
        for c_id in unique_clusters:
            indices = np.where(cluster_labels == c_id)[0]
            aligned_patches = []
            for i in indices:
                cy, cx = region_data[i].centroid; tr, orr = int(self.particle_radius*1.1), int(self.particle_radius*1.6)
                if 0 < int(cy)-orr and int(cy)+orr < prob_map.shape[0] and 0 < int(cx)-orr and int(cx)+orr < prob_map.shape[1]:
                    p = rotate(micrograph[int(cy)-orr:int(cy)+orr, int(cx)-orr:int(cx)+orr], -np.degrees(region_data[i].orientation), reshape=False, order=1)
                    aligned_patches.append(p[orr-tr:orr+tr, orr-tr:orr+tr])
            if aligned_patches:
                mean_img = np.mean(aligned_patches, axis=0)
                snr_prof, shell_bounds = self._calculate_snr_profile(mean_img)
                aligned_data[c_id] = (mean_img, snr_prof, shell_bounds)
                # Call quality evaluator with Levene's variance test
                status, m_mean = self._evaluate_cluster_quality(c_id, indices, all_mahal, X, cluster_labels, mahal_threshold)
                cluster_outlier_status[c_id], cluster_mahal_means[c_id] = status, m_mean

        if show_plots:
            self.plot_diagnostics(micrograph, prob_map, region_data, unique_clusters, cluster_outlier_status, cluster_mahal_means, X, all_mahal, cluster_labels, aligned_data, X_emb, graph_name)

        valid_particles = []
        for c_id in unique_clusters:
            if cluster_outlier_status[c_id] == "No":
                c_idx = np.where(cluster_labels == c_id)[0]
                candidates = [[int(region_data[i].centroid[1]), int(region_data[i].centroid[0]), float(prob_map[int(region_data[i].centroid[0]), int(region_data[i].centroid[1])])]
                              for i in c_idx if 40 < region_data[i].centroid[1] < prob_map.shape[1] - 40 and 40 < region_data[i].centroid[0] < prob_map.shape[0] - 40]
                valid_particles.extend(random.sample(candidates, min(len(candidates), N_from_clust)))
        return valid_particles

In [51]:
import itertools
import random

#  Instantiate the pipeline class once
pipeline = Watershed_Cluster(particle_radius=RADIUS, n_clusters=7, seed=42)

watershed_list_all = []
watershed_config = []

# Search Space (can be expanded)
threshold_levels_ratios = [0.4]
min_dist_ratios = [0.8]

print(f"Starting DT-Only Grid Search with Class-based Pipeline...")

param_grid = list(itertools.product(threshold_levels_ratios, min_dist_ratios))

# Iterate through hyperparameters
for params in param_grid:
    thresh_r, dist_r = params
    print(f"\n--- Testing Config: Thresh={thresh_r}, Dist={dist_r} ---")
    current_config_results = []

    # Iterate through validation dataset
    for idx in range(len(val_filenames)):
        name = val_filenames[idx][:-4]

        # 1. Load data
        print(f"\n[Processing Graph Index: {idx}] Micrograph: {name}")
        micrograph = np.load(f"{IMAGE_DIR}/val/{name}.npy")
        prob_map = label_images[idx]

        # 2. Match dimensions (preprocessing)
        cropped_micrograph = preprocess_and_crop(micrograph)

        # 3. Perform Modular Pipeline
        particles = pipeline.process(
            micrograph_input=cropped_micrograph,
            prob_map_input=prob_map,
            peak_threshold_ratio=thresh_r,
            min_dist_ratio=dist_r,
            N_from_clust=10,
            show_plots=True,
            graph_name=name
        )
        current_config_results.append(particles)

    watershed_list_all.append(current_config_results)
    watershed_config.append((thresh_r, dist_r))

print(f"\nGrid Search Complete. Evaluated {len(watershed_config)} candidates.")

Output hidden; open in https://colab.research.google.com to view.

In [52]:
# @title DT-Only Grid Search: Evaluate & Score
import torch
import os
import json

# Metrics Helper
def evaluate_detection_robust(iou_matrices, iou_threshold=0.5):
    tp_total = 0; fp_total = 0; fn_total = 0
    for iou_matrix in iou_matrices:
        n_gt, n_pred = iou_matrix.shape
        if n_gt == 0 and n_pred == 0: continue
        if n_gt > 0 and n_pred == 0: fn_total += n_gt; continue
        if n_gt == 0 and n_pred > 0: fp_total += n_pred; continue

        max_vals, _ = torch.max(iou_matrix, dim=1)
        detected_mask = max_vals >= iou_threshold
        tp = torch.sum(detected_mask).item()
        tp_total += tp
        fn_total += (n_gt - tp)
        fp_total += (n_pred - tp)

    epsilon = 1e-6
    precision = tp_total / (tp_total + fp_total + epsilon)
    recall = tp_total / (tp_total + fn_total + epsilon)
    return precision, recall

# Main Eval Loop
width = RADIUS * 2
watershed_scores = []

print("Evaluating DT-Only Results...")

for i, ws_list in enumerate(watershed_list_all):
    cfg = watershed_config[i] # (thresh, dist)

    iou_matrices = []
    gt_full = []
    pred_full = []
    scores_list = []

    for idx, gt in enumerate(gts):
        gt_boxes = centers_to_boxes(np.array(gt), width, width)
        gt_full.append(gt_boxes)

        preds = ws_list[idx]
        if len(preds) > 0:
            coords = np.array([p[:2] for p in preds])
            sc = torch.tensor([p[2] for p in preds], dtype=torch.float32)
            pred_boxes = centers_to_boxes(coords, width, width)
        else:
            pred_boxes = torch.empty((0, 4))
            sc = torch.tensor([], dtype=torch.float32)

        pred_full.append(pred_boxes)
        scores_list.append(sc)

        iou_matrix = calculate_iou_torchvision(gt_boxes, pred_boxes)
        iou_matrices.append(iou_matrix)

    # 1. F-Beta
    precision, recall = evaluate_detection_robust(iou_matrices, iou_threshold=0.5)
    f_beta = f_beta_score(precision, recall, beta=1)

    # 2. mAP
    iou_thresholds = torch.arange(0.5, 1.0, 0.05)
    try:
        mAP_value = calculate_mAP_multiple_images(gt_full, pred_full, scores_list, iou_thresholds)
        map_val = mAP_value.item()
    except:
        map_val = 0.0

    print(f"Cfg: {cfg} | F1: {f_beta:.4f}, mAP: {map_val:.4f}")
    watershed_scores.append((f_beta, map_val))

# Save Best
if len(watershed_scores) > 0:
    best_idx = max(range(len(watershed_scores)), key=lambda i: watershed_scores[i][0])
    best_cfg = watershed_config[best_idx]
    method = "DT_cluster_class" # @param{type:"string"}
    best_res = {
        "method": method,
        "radius": RADIUS,
        "peak_thresh": best_cfg[0],
        "min_dist": best_cfg[1],
        "f_score": watershed_scores[best_idx][0],
        "map": watershed_scores[best_idx][1]
    }

    print("\n--- BEST CONFIGURATION (DT ONLY - NO AR) ---")
    print(best_res)

    # 1. Save to local content first (Current Directory)
    local_file = 'best_watershed_cluster_class_params.json' # @param{type:"string"}
    with open(local_file, 'w') as f:
        json.dump(best_res, f, indent=4)
    print(f"✅ Saved locally to: {local_file}")

    # 2. Copy to RESULT_DIR using shell command
    # We put quotes "" around the path in case RESULT_DIR has spaces
    !cp {local_file} "{RESULT_DIR}"

    print(f"✅ Copied to: {RESULT_DIR}/{local_file}")

Evaluating DT-Only Results...
Cfg: (0.4, 0.8) | F1: 0.1608, mAP: 0.0904

--- BEST CONFIGURATION (DT ONLY - NO AR) ---
{'method': 'DT_cluster_class', 'radius': 64, 'peak_thresh': 0.4, 'min_dist': 0.8, 'f_score': 0.16083009070774057, 'map': 0.09040865302085876}
✅ Saved locally to: best_watershed_cluster_class_params.json
✅ Copied to: /content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output_rst/results_watershed_clustering/10017/unet_eb5_dice_CRF/best_watershed_cluster_class_params.json
